# Stage B — Analysis & Evaluation

Rigid-sphere indentation. FEM penalty contact (tet penalty sweep + hex) vs
Newton (XPBD). Evaluates contact force, deformed dimple, internal & contact
energy, penetration, computation time. **Friction is not included yet** (both
runs frictionless) - see the last section.

In [ ]:
%matplotlib inline
import os
import numpy as np
import matplotlib.pyplot as plt
from common import params

fem = np.load(params.FEM_STAGEB_NPZ)
newton = np.load(params.NEWTON_STAGEB_NPZ) if os.path.exists(params.NEWTON_STAGEB_NPZ) else None
slugs = [k[3:] for k in fem.files if k.startswith('uz_')]
d = fem['deltas'] * 1000.0
print('FEM variants:', slugs, '| Newton:', newton is not None)

## 1. Contact force vs indentation (FEM) with Hertz anchor

In [ ]:
plt.figure(figsize=(6, 5))
for s in slugs:
    plt.plot(d, fem['f_' + s], 'o-', label='FEM ' + s)
plt.plot(d, fem['f_hertz'], 'k--', lw=1.5, label='Hertz')
plt.xlabel('indentation [mm]'); plt.ylabel('contact force [N]'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Stage B - contact force (XPBD has no comparable force)'); plt.show()

## 2. Deformed dimple - FEM vs Newton

In [ ]:
plt.figure(figsize=(6, 5))
fx = (fem['line_x'] - float(fem['cx'])) * 1000.0
for s in slugs:
    plt.plot(fx, fem['uz_' + s] * 1000.0, label='FEM ' + s)
if newton is not None:
    nx = (newton['line_x'] - float(newton['cx'])) * 1000.0
    plt.plot(nx, newton['uz_line'] * 1000.0, 'k-o', ms=3, label='Newton XPBD')
plt.xlabel('x - centre [mm]'); plt.ylabel('u_z [mm]'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Stage B - deformed dimple'); plt.show()

## 3. Internal (strain) energy vs indentation

In [ ]:
plt.figure(figsize=(6, 5))
for s in slugs:
    plt.plot(d, fem['e_strain_' + s], 'o-', label='FEM ' + s)
if newton is not None:
    plt.plot(d, newton['e_strain'], 'k-o', ms=3, label='Newton XPBD')
plt.xlabel('indentation [mm]'); plt.ylabel('strain energy [J]'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Stage B - internal energy'); plt.show()

## 4. Contact (penalty) energy - FEM\n\nPenalty potential 1/2 kn <-g>+^2 over the contact surface. Stiffer kn stores less of it (less penetration).

In [ ]:
plt.figure(figsize=(6, 5))
for s in slugs:
    plt.plot(d, fem['e_contact_' + s], 'o-', label='FEM ' + s)
plt.xlabel('indentation [mm]'); plt.ylabel('contact (penalty) energy [J]'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Stage B - contact energy (FEM penalty)'); plt.show()

## 5. Newton - penetration & kinetic-energy (settle) diagnostic

In [ ]:
if newton is not None:
    fig, ax1 = plt.subplots(figsize=(6, 4))
    ax1.plot(d, newton['penetration'] * 1000.0, 'tab:red', marker='o', ms=3)
    ax1.set_xlabel('indentation [mm]'); ax1.set_ylabel('max penetration [mm]', color='tab:red')
    ax2 = ax1.twinx(); ax2.semilogy(d, np.maximum(newton['ke'], 1e-12), 'tab:green', alpha=0.7)
    ax2.set_ylabel('residual KE [J] (settled ~ 0)', color='tab:green')
    plt.title('Stage B - Newton penetration & settle KE'); plt.tight_layout(); plt.show()
else:
    print('No Newton Stage B result yet (run newton_run.run_stage_b).')

## 6. Computation time (wall-clock)

In [ ]:
ft = {s: float(fem['time_' + s]) for s in slugs if ('time_' + s) in fem.files}
if newton is not None and 'wall_time' in newton.files:
    ft['Newton XPBD'] = float(newton['wall_time'])
for k, v in ft.items():
    print(f'{k:12s}: {v:8.3f} s')
plt.figure(figsize=(6, 4))
plt.bar(list(ft), list(ft.values()))
plt.ylabel('wall time [s]'); plt.title('Stage B - computation time'); plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## 7. FEM residual penetration: penalty vs Augmented Lagrangian

The Augmented-Lagrangian payoff: at the *same* modest `kn`, the Uzawa multiplier
loop drives the residual penetration toward zero (approaching the exact
non-penetration constraint), while plain penalty leaves a kn-dependent
penetration. Compare `tet kn x5 penalty` vs `tet kn x5 AL`.

In [ ]:
plt.figure(figsize=(6, 5))
for s in slugs:
    if ('pen_' + s) in fem.files:
        plt.plot(d, fem['pen_' + s] * 1000.0, 'o-', label='FEM ' + s)
plt.xlabel('indentation [mm]'); plt.ylabel('max penetration [mm]'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Stage B - residual penetration (penalty vs AL)'); plt.show()

## 8. Friction — moved to its own scenario

Stage B (the rigid-sphere indentation) stays **frictionless** on purpose, so the
Hertz normal comparison is clean. Coulomb friction — **friction force and dissipated
work** — is now a dedicated sliding-block scenario:

* FEM: `python -m fenics_run.run_friction` (penalty-regularised Coulomb, return mapping)
* Newton: `python -m newton_run.run_friction` (`soft_contact_mu`, kinematic drag)
* Analysis: **`40_friction_analysis.ipynb`** — friction force vs the analytic `mu*W`
  plateau, dissipated work, and the stick→slip transition.

As with the normal contact force, XPBD exposes the slip but no calibrated friction
force; the FEM run provides both.